# Somerville Software Engineering RAG Project - Q&A App using LangChain, OpenAI and Pinecone

### Requirements and Environment Setup

In [1]:
pip install -q -r ./requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

# .env files are not included in the repository for security reasons (as specified
# in the .gitignore file), so create your own .env file with the following content:
# OPENAI_API_KEY=""
# PINECONE_API_KEY=""
# LANGCHAIN_API_KEY=""
# LANGCHAIN_TRACING_V2="false"

True

### Loading Documents

In [3]:
# loading PDF, DOCX and TXT files as LangChain Documents
def load_document(file):
    import os
    name, extension = os.path.splitext(file)

    # if extension == '.pdf':
    #     from langchain_pymupdf4llm import PyMuPDF4LLMLoader
    #     os.environ['TESSDATA_PREFIX'] = "./tessdata"
    #     print(f'Loading {file} with OCR enabled via data path: {os.environ['TESSDATA_PREFIX']}')
    #     loader = PyMuPDF4LLMLoader(file)
    if extension == '.pdf':
        from langchain_classic.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader = PyPDFLoader(file)
    elif extension == '.docx':
        from langchain_classic.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader = Docx2txtLoader(file)
    elif extension == '.txt':
        from langchain_classic.document_loaders import TextLoader
        loader = TextLoader(file)
        print(f'Loading {file}')
    else:
        print('Document format is not supported!')
        return None

    data = loader.load()
    return data

### Chunking Data

In [4]:
def chunk_data(data, chunk_size=256):
    from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=0)
    chunks = text_splitter.split_documents(data)
    return chunks

### Calculating Cost

In [5]:
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    # check prices here: https://openai.com/pricing
    print(f'Total Tokens: {total_tokens}')
    print(f'Embedding Cost in USD: {total_tokens / 1000 * 0.00002:.6f}')

### Embedding and Uploading to the Pinecone Vector Database

In [6]:
# - give every chunk a deterministic Pinecone ID based on
# - its document source, page, page label and chunk text
# - check each chunk individually
# - skip chunks for which ID already exists in Pinecone
# - only insert new chunks

def insert_or_fetch_embeddings(index_name, chunks, namespace):
    """Insert only new embeddings into Pinecone, then return the vector store."""
    import hashlib
    import time
    import pinecone
    from langchain_community.vectorstores import Pinecone as PineconeVectorStore
    from langchain_openai import OpenAIEmbeddings
    from pinecone import ServerlessSpec

    def make_chunk_id(chunk):
        source = str(chunk.metadata.get("source", "unknown-source")).replace("\\", "/")
        page = str(chunk.metadata.get("page", "unknown-page"))
        page_label = str(chunk.metadata.get("page_label", "unknown-page-label"))
        id_text = f"{source}|{page}|{page_label}|{chunk.page_content}"
        return hashlib.sha256(id_text.encode("utf-8")).hexdigest()

    pc = pinecone.Pinecone()
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)

    existing_indexes = [idx.name for idx in pc.list_indexes()]

    if index_name not in existing_indexes:
        print(f"Creating index {index_name} ... ", end="")

        pc.create_index(
            name=index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )

        while not pc.describe_index(index_name).status["ready"]:
            time.sleep(1)

        print("Ok")

    pinecone_index = pc.Index(index_name)

    vector_store = PineconeVectorStore(
        index=pinecone_index,
        embedding=embeddings,
        text_key="text",
        namespace=namespace
    )

    chunk_ids = [make_chunk_id(chunk) for chunk in chunks]
    chunks_to_add = []
    ids_to_add = []

    fetch_batch_size = 100

    for i in range(0, len(chunk_ids), fetch_batch_size):
        batch_ids = chunk_ids[i:i + fetch_batch_size]
        batch_chunks = chunks[i:i + fetch_batch_size]

        fetch_response = pinecone_index.fetch(
            ids=batch_ids,
            namespace=namespace
        )

        existing_vectors = getattr(fetch_response, "vectors", None)

        if existing_vectors is None:
            existing_vectors = fetch_response.get("vectors", {})

        existing_ids = set(existing_vectors.keys())

        for chunk_id, chunk in zip(batch_ids, batch_chunks):
            if chunk_id not in existing_ids:
                ids_to_add.append(chunk_id)
                chunks_to_add.append(chunk)

    if chunks_to_add:
        print(f"Adding {len(chunks_to_add)} new documents to index {index_name} ...\n", end="")

        embedding_batch_size = 100
        total_batches = (len(chunks_to_add) + embedding_batch_size - 1) // embedding_batch_size

        for i in range(0, len(chunks_to_add), embedding_batch_size):
            batch_chunks = chunks_to_add[i:i + embedding_batch_size]
            batch_ids = ids_to_add[i:i + embedding_batch_size]
            batch_texts = [chunk.page_content for chunk in batch_chunks]

            batch_number = (i // embedding_batch_size) + 1
            print(f"Adding batch {batch_number}/{total_batches} ({len(batch_chunks)} documents) ... ", end="")

            batch_embeddings = embeddings.embed_documents(batch_texts)

            vectors = []

            for chunk_id, chunk, embedding in zip(batch_ids, batch_chunks, batch_embeddings):
                metadata = dict(chunk.metadata)
                metadata["text"] = chunk.page_content

                vectors.append({
                    "id": chunk_id,
                    "values": embedding,
                    "metadata": metadata
                })

            pinecone_index.upsert(
                vectors=vectors,
                namespace=namespace
            )

            print("Ok")

        print("All batches added successfully!")

        # Give Pinecone a moment to make freshly upserted vectors visible to fetch/stats.
        time.sleep(5)
    else:
        print(f"All {len(chunks)} documents already exist in index {index_name}. No new documents added.")

    return vector_store

In [7]:
def delete_pinecone_index(index_name='all'):
    import pinecone
    pc = pinecone.Pinecone()
    
    if index_name == 'all':
        indexes = pc.list_indexes().names()
        print('Deleting all indexes ... ')
        for index in indexes:
            pc.delete_index(index)
        print('OK')
    else:
        print(f'Deleting index {index_name} ...', end='')
        pc.delete_index(index_name)
        print('OK')

### Asking and Getting Answers

In [25]:
def ask_and_get_answer(vector_store, q, k=10):
    from langchain_classic.chains import RetrievalQA
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model='gpt-4o-mini', temperature=1)

    retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': k})

    chain = RetrievalQA.from_chain_type(llm=llm,
                                        chain_type="stuff",
                                        retriever=retriever,
                                        return_source_documents=True)
    
    answer = chain.invoke(q)
    return answer

### Running Code

#### Load Documents and Chunk Data

In [11]:
from pathlib import Path

# amend accordingly
folder_paths = [
    Path("../Somerville_Software_Engineering_Docs"),
]

data = []

for folder_path in folder_paths:
    for file_path in folder_path.iterdir():
        if file_path.suffix.lower() in [".pdf", ".docx", ".txt"]:
            document_pages = load_document(str(file_path))

            if document_pages is not None:
                for page in document_pages:
                    page.metadata["page_label"] = page.metadata.get("page_label", "Unknown page")
                    page.metadata["page"] = page.metadata.get("page", "Unknown page")

                data.extend(document_pages)

print(f"There are {len(data)} pages in total")

if data:
    print(data[0].page_content)
    print(data[0].metadata)
    print(f"There are {len(data[0].page_content)} characters on the first page")

Loading ..\Somerville_Software_Engineering_Docs\se_book_tenth_edition_software_engineering_ian_sommerville.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch10_dependable_systems.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch11_reliability_engineering.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch12_safety_engineering.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch13_security_engineering.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch14_resilience_engineering.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch15_software_reuse.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch16_component-based_software_engineering.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch17_distributed_software_engineering.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch18_service-oriented_software_engineering.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch19_systems_engineering.pdf
Loading ..\Somerville_Software_Engineeri

Ignoring wrong pointing object 83 0 (offset 0)


Loading ..\Somerville_Software_Engineering_Docs\se_ch26_process_improvement.pdf


Ignoring wrong pointing object 67 0 (offset 0)
Ignoring wrong pointing object 78 0 (offset 0)
Ignoring wrong pointing object 87 0 (offset 0)
Ignoring wrong pointing object 101 0 (offset 0)
Ignoring wrong pointing object 120 0 (offset 0)


Loading ..\Somerville_Software_Engineering_Docs\se_ch27_formal_specification.pdf


...

Loading ..\Somerville_Software_Engineering_Docs\se_ch28_application_architecture.pdf


Ignoring wrong pointing object 147 0 (offset 0)


Loading ..\Somerville_Software_Engineering_Docs\se_ch29_interaction_design.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch2_software_processes.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch30_documentation.pdf


Ignoring wrong pointing object 125 0 (offset 0)


Loading ..\Somerville_Software_Engineering_Docs\se_ch31_aspect-oriented_software_engineering.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch3_agile_software_development.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch4_requirements_engineering.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch5_system_modeling.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch6_architectural_design.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch7_design_and_implementation.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch8_software_testing.pdf
Loading ..\Somerville_Software_Engineering_Docs\se_ch9_software_evolution.pdf
There are 2570 pages in total
Glob Al  
Edi Tio N
Software Engineering
TENTH Edi Tio N 
ian Sommerville
{'producer': 'www.ilovepdf.com', 'creator': '-|-  this layout: dynstab  -|-', 'creationdate': '2017-09-18T09:08:24-04:00', 'title': 'Software Engineering, 10th GLOBAL Edition', 'author': 'Ian Sommerville', 'subject': '', 'keywords': ''

In [12]:
chunks = chunk_data(data)
print(len(chunks))
print(chunks[len(chunks) - 1].page_content)

15004
Chapter 9 Software Evolution 6130/10/2014


In [13]:
print_embedding_cost(chunks)

Total Tokens: 665236
Embedding Cost in USD: 0.013305


In [16]:
import pinecone
pc = pinecone.Pinecone()
pc.list_indexes()

IndexList([<name='som-soft-eng-rag', dim=1536, ready=True>, <name='luv2code-rag', dim=1536, ready=True>, <name='som-esp-rag', dim=1536, ready=True>, <name='fastapi-rag', dim=1536, ready=True>, <name='ou-comp-it-degree', dim=1536, ready=True>])

In [15]:
from pinecone import ServerlessSpec
index_name = 'som-soft-eng-rag'
if index_name not in pc.list_indexes().names():
    print(f'Creating index: {index_name} ...', end='')
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print('Index created!')
else:
    print(f'Index {index_name} already exists!')
# pc.list_indexes()[0]['name']
pc.describe_index('som-soft-eng-rag')

Creating index: som-soft-eng-rag ...Index created!


IndexModel(name='som-soft-eng-rag', metric='cosine', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://som-soft-eng-rag-moalxso.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=1536, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [20]:
index_name = 'som-soft-eng-rag'
index = pc.Index(index_name)
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=1536, total_vector_count=15004, metric='cosine', namespaces=1)

In [18]:
# Only use this if you need to
# delete_pinecone_index('som-soft-eng-rag')

In [21]:
index_name = 'som-soft-eng-rag'
namespace = 'som-soft-eng-rag'
vector_store = insert_or_fetch_embeddings(index_name=index_name, chunks=chunks, namespace=namespace)

All 15004 documents already exist in index som-soft-eng-rag. No new documents added.


In [26]:
q = 'Write four paragraphs about software testing. Answer from the provided context only.'
answer = ask_and_get_answer(vector_store, q)
print(answer['result'])
print("\nReference(s):")

for doc in answer["source_documents"]:
    source = doc.metadata.get("source", "Unknown document")
    page_label = doc.metadata.get("page_label")
    page = doc.metadata.get("page")

    print(f"\nDocument source: {source}")
    print(f"Page label: {page_label}")
    print(f"Page index: {page}")

Software testing is a crucial part of the software development process, ensuring that the product meets the required standards before it is released. It involves systematically evaluating different aspects of the software to identify defects and validate that the software functions as intended. The testing process can vary in complexity, depending on the criticality of the system being developed. For critical systems, a more formal and structured testing process may be necessary, often involving a separate testing group within the development team to ensure a comprehensive review.

An important aspect of software testing is that it should always be an incremental, staged activity. This approach allows for the identification and resolution of issues at various stages of development, rather than trying to address all problems at once. By breaking down the testing into smaller, manageable stages, teams can focus on specific functionalities, making it easier to isolate and fix errors befor

#### While Loop for Asking Questions

In [28]:
import time
i = 1
print('Enter your questions (or write Quit to quit).')

while True:
    q = input(f'Question #{i}: ') + ' Answer from the provided context only.'
    i = i + 1
    if q.lower().startswith('quit'):
        print('\nQuitting ... bye bye!')
        time.sleep(2)
        break
    
    answer = ask_and_get_answer(vector_store, q)
    print(f'\n{"-" * 50}')
    print(f'\nQuery: ' + answer['query'])
    print(f'\nAnswer: ' + answer['result'])
    print("\nReference(s):")

    for doc in answer["source_documents"]:
        source = doc.metadata.get("source", "Unknown document")
        page_label = doc.metadata.get("page_label")
        page = doc.metadata.get("page")

        print(f"\nDocument: {source}")
        print(f"Page label: {page_label}")
        print(f"Page index: {page}")


Enter your questions (or write Quit to quit).

--------------------------------------------------

Query: Write four paragraphs about system modeling. Answer from the provided context only.

Answer: System modeling is the process of developing abstract representations of a system, offering various perspectives and views. This approach allows for different aspects of a system to be analyzed and understood independently. By creating models, stakeholders can visualize how components of the system interact, as well as identify potential issues or improvements. The diversity in models aids in covering multiple facets of the system, which is crucial for effective system design and analysis.

Context models are one type of system model that help to define the environment in which the system operates. They provide a framework to understand the interactions between the system and its external entities. This can include users, hardware, and other systems that may influence or be influenced by th